# Hybrid Machine Learning and Optimized ANFIS Ensemble for Software Defect Prediction

Research-grade cross-project software defect prediction using NASA MDP datasets **CM1, JM1, KC1, KC3 and PC1**.

The final predictive architecture contains exactly six models:

1. Random Forest
2. XGBoost
3. LightGBM
4. CatBoost
5. Optimized Sugeno ANFIS
6. Hybrid ML + ANFIS Ensemble

The experiment uses leakage-safe Leave-One-Dataset-Out (LODO) evaluation. Model selection, feature selection, preprocessing, threshold selection and ensemble weighting are performed using training data and an internal validation split only. The held-out project is evaluated only after all decisions are frozen.

The notebook is designed to be executable from top to bottom in Google Colab or Jupyter. It exports reproducible tables, plots, statistical tests, bootstrap intervals, ablation results, feature importance and serialized final models.

## 1. Project Overview

**Research objective:** improve defective-class detection while preserving a strict cross-project evaluation protocol. Primary comparison metrics are F1, Recall, ROC-AUC and PR-AUC; Accuracy, MAE, MSE and R² are retained as supplementary measures.

## 2. Imports and Configuration

In [ ]:
!pip -q install xgboost lightgbm catboost scipy statsmodels scikit-learn pandas numpy matplotlib seaborn joblib

import os, io, re, zipfile, json, copy, random, itertools, warnings, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import torch
import torch.nn as nn

from scipy.io import arff
from scipy import stats
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.model_selection import train_test_split, ParameterSampler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, balanced_accuracy_score, confusion_matrix,
    roc_curve, precision_recall_curve, mean_absolute_error, mean_squared_error,
    r2_score, matthews_corrcoef
)
from statsmodels.stats.multitest import multipletests
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, early_stopping
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
STABILITY_SEEDS = [42, 52, 62, 72, 82]
SELECTED_DATASETS = ["CM1", "JM1", "KC1", "KC3", "PC1"]
# Wider feature-count search. The selector automatically caps k at the available feature count.
FEATURE_COUNTS = [5, 8, 10, 12, 16, 20, 24, 32]
# Controlled ANFIS rule base: 4 inputs x 3 MFs = 81 rules maximum.
ANFIS_MAX_INPUTS = 4
VALIDATION_SIZE = 0.25
THRESHOLD_GRID = np.round(np.arange(0.02, 0.981, 0.01), 2)
TREE_N_ITER = 24

# Multi-objective validation: improve classification quality without optimizing only one metric.
# MCC is rescaled from [-1,1] to [0,1].
OBJECTIVE_WEIGHTS = {"Accuracy":0.65, "F1":0.15, "Balanced_Accuracy":0.15, "MCC_scaled":0.05}

RESULTS_DIR = Path("results_v7_accuracy_priority")
for sub in ["tables", "plots", "confusion_matrices", "roc_curves", "pr_curves",
            "feature_importance", "statistical_tests", "bootstrap", "ablation", "models"]:
    (RESULTS_DIR / sub).mkdir(parents=True, exist_ok=True)

def set_global_seed(seed=RANDOM_STATE):
    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_global_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAMES = ["Random Forest", "XGBoost", "LightGBM", "CatBoost", "ANFIS", "Hybrid ML + ANFIS"]
assert len(MODEL_NAMES) == 6 and len(set(MODEL_NAMES)) == 6
PROJECT_TITLE = "Hybrid Machine Learning and Optimized ANFIS Ensemble for Software Defect Prediction — Accuracy-Priority v7 — 21-Feature Compatible — Research-Validated"
print(PROJECT_TITLE)
print("Device:", DEVICE)
print("Models:", MODEL_NAMES)
print("TREE_N_ITER:", TREE_N_ITER)
V7_CONFIG = {"version":"v7","accuracy_priority":True,"cluster_bootstrap":True,"max_anfis_rules":81}
print("V7 configuration:", V7_CONFIG)
print("Objective weights (accuracy-priority):", OBJECTIVE_WEIGHTS)


## 3. Dataset Loading

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
except Exception:
    uploaded = {}
    print("Colab upload widget unavailable. Use LOCAL_DATA_PATH below for Jupyter.")

LOCAL_DATA_PATH = ""

def _dataset_priority(member_name):
    s = member_name.replace("\\", "/").lower()
    if "cleaneddata/mdp/d''/" in s: return 0
    if "cleaneddata/mdp/d'/" in s: return 1
    if "originaldata/mdp/" in s: return 2
    return 3

def read_arff_bytes(raw_bytes, dataset_name):
    try:
        text = raw_bytes.decode("utf-8")
    except UnicodeDecodeError:
        text = raw_bytes.decode("latin-1")
    data_arr, meta = arff.loadarff(io.StringIO(text))
    df = pd.DataFrame(data_arr)
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].apply(lambda x: x.decode("utf-8", "ignore") if isinstance(x, (bytes, bytearray)) else x)
    target_col = df.columns[-1]
    target = df[target_col].astype(str).str.strip().str.lower().str.rstrip(".")
    positive = {"y", "yes", "true", "1", "defective", "buggy"}
    df["defects"] = target.isin(positive).astype(int)
    predictors = []
    for col in df.columns[:-1]:
        numeric = pd.to_numeric(df[col], errors="coerce")
        if numeric.notna().sum() > 0:
            df[col] = numeric
            predictors.append(col)
    df = df[predictors + ["defects"]].copy()
    df = df.replace([np.inf, -np.inf], np.nan)
    df["dataset"] = dataset_name
    return df

def collect_arffs(source_dict):
    records = []
    for name, raw in source_dict.items():
        if name.lower().endswith(".arff"):
            records.append((name, raw))
        elif name.lower().endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(raw), "r") as z:
                for member in z.namelist():
                    if member.lower().endswith(".arff") and not member.endswith("/"):
                        records.append((member, z.read(member)))
    return records

source = dict(uploaded)
if LOCAL_DATA_PATH:
    p = Path(LOCAL_DATA_PATH)
    if p.suffix.lower() in {".zip", ".arff"}:
        source[p.name] = p.read_bytes()

records = collect_arffs(source)
if not records:
    raise FileNotFoundError("No ARFF files found. Upload the NASA MDP ZIP/ARFF files or set LOCAL_DATA_PATH.")

candidates = {}
for member, raw in records:
    base = Path(member).stem.upper()
    if base in SELECTED_DATASETS:
        candidates.setdefault(base, []).append((member, raw))

dataset_frames, chosen_files = {}, {}
for d in SELECTED_DATASETS:
    opts = sorted(candidates.get(d, []), key=lambda x: _dataset_priority(x[0]))
    if not opts:
        raise ValueError(f"Required dataset {d} was not found.")
    member, raw = opts[0]
    dataset_frames[d] = read_arff_bytes(raw, d)
    chosen_files[d] = member

common = set(dataset_frames[SELECTED_DATASETS[0]].columns) - {"defects", "dataset"}
for d in SELECTED_DATASETS[1:]:
    common &= set(dataset_frames[d].columns) - {"defects", "dataset"}
COMMON_FEATURES = sorted(common)

# NASA MDP projects do not necessarily share every metric. The requested
# feature counts are therefore capped at the number of common features
# actually available, instead of stopping with an error.
if len(COMMON_FEATURES) == 0:
    raise ValueError("No common numeric features were found across the selected NASA datasets.")
EFFECTIVE_FEATURE_COUNTS = sorted(set(min(int(k), len(COMMON_FEATURES)) for k in FEATURE_COUNTS))


data = pd.concat([dataset_frames[d][COMMON_FEATURES + ["defects", "dataset"]].copy() for d in SELECTED_DATASETS], ignore_index=True)
data["defects"] = data["defects"].astype(int)

print("Selected files:")
for d in SELECTED_DATASETS: print(f"  {d}: {chosen_files[d]}")
print(f"Rows: {len(data):,}")
print(f"Common numeric features: {len(COMMON_FEATURES)}")
print(data["defects"].value_counts().rename({0:"non-defective",1:"defective"}))
print(data["dataset"].value_counts())

### Common-feature compatibility fix
The NASA datasets in the uploaded archive share 21 common numeric features. The pipeline now automatically caps requested feature counts at the available common-feature count, so a 32-feature request will not stop execution. This does not add synthetic or duplicated predictors.


## 4. Dataset Inspection

In [ ]:
inspection = []
for d in SELECTED_DATASETS:
    frame = data[data["dataset"] == d]
    inspection.append({
        "Dataset": d, "Rows": len(frame), "Features": len(COMMON_FEATURES),
        "Defective": int(frame.defects.sum()), "Non_Defective": int((frame.defects == 0).sum()),
        "Defect_Rate": float(frame.defects.mean()),
        "Duplicate_Rows": int(frame[COMMON_FEATURES + ["defects"]].duplicated().sum()),
        "Infinite_Values": int(np.isinf(frame[COMMON_FEATURES].to_numpy(dtype=float)).sum())
    })
inspection_df = pd.DataFrame(inspection)
display(inspection_df)
missing_report = data[COMMON_FEATURES].isna().mean().sort_values(ascending=False).rename("missing_fraction").to_frame()
display(missing_report.head(15))

## 5. Data Cleaning

In [ ]:
X_all = data[COMMON_FEATURES].copy().replace([np.inf, -np.inf], np.nan)
y_all = data["defects"].copy()
global_constant = [c for c in COMMON_FEATURES if X_all[c].nunique(dropna=True) <= 1]
FEATURE_POOL = [c for c in COMMON_FEATURES if c not in global_constant]
print("Globally constant features removed:", global_constant if global_constant else "None")
print("Feature pool:", len(FEATURE_POOL))
if len(FEATURE_POOL) == 0:
    raise ValueError("All common numeric features are constant; no usable predictors remain.")
EFFECTIVE_FEATURE_COUNTS = sorted(set(min(int(k), len(FEATURE_POOL)) for k in FEATURE_COUNTS))
print("Effective feature-count search:", EFFECTIVE_FEATURE_COUNTS)

## 6. Leakage-Safe Preprocessing

In [ ]:
def make_internal_split(train_df):
    tr, va = train_test_split(train_df, test_size=VALIDATION_SIZE,
                               stratify=train_df["defects"], random_state=RANDOM_STATE)
    return tr.reset_index(drop=True), va.reset_index(drop=True)

def validation_objective(y_true, proba, threshold):
    pred = (np.asarray(proba) >= threshold).astype(int)
    acc = accuracy_score(y_true, pred)
    f1 = f1_score(y_true, pred, zero_division=0)
    bal = balanced_accuracy_score(y_true, pred)
    mcc = matthews_corrcoef(y_true, pred)
    mcc_scaled = (mcc + 1.0) / 2.0
    score = (OBJECTIVE_WEIGHTS["Accuracy"]*acc +
             OBJECTIVE_WEIGHTS["F1"]*f1 +
             OBJECTIVE_WEIGHTS["Balanced_Accuracy"]*bal +
             OBJECTIVE_WEIGHTS["MCC_scaled"]*mcc_scaled)
    return float(score), {"Accuracy":acc,"F1":f1,"Balanced_Accuracy":bal,"MCC":mcc}

def optimize_threshold(y_true, proba):
    best_t, best_score, best_detail = 0.50, -np.inf, None
    for t in THRESHOLD_GRID:
        score, detail = validation_objective(y_true, proba, t)
        # Accuracy is the primary research target; F1/balanced accuracy are tie-breakers.
        tie_key = (round(score, 10), detail["Accuracy"], detail["F1"], detail["Balanced_Accuracy"], detail["MCC"])
        if best_detail is None or tie_key > best_detail["tie_key"]:
            best_score = score
            best_t = float(t)
            best_detail = {**detail, "tie_key": tie_key}
    return best_t, best_score

def select_features_from_training(X_fit, y_fit, X_eval, feature_names, k):
    k = min(int(k), X_fit.shape[1])
    selector = SelectKBest(score_func=lambda X,y: mutual_info_classif(X,y,random_state=RANDOM_STATE), k=k)
    A = selector.fit_transform(X_fit, y_fit)
    B = selector.transform(X_eval)
    selected = list(np.asarray(feature_names)[selector.get_support()])
    scores = pd.Series(selector.scores_, index=feature_names).sort_values(ascending=False)
    return selector, A, B, selected, scores

def choose_feature_count(train_df):
    inner_tr, inner_va = make_internal_split(train_df)
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    A = scaler.fit_transform(imputer.fit_transform(inner_tr[FEATURE_POOL]))
    B = scaler.transform(imputer.transform(inner_va[FEATURE_POOL]))
    rows = []
    for requested_k in EFFECTIVE_FEATURE_COUNTS:
        k = min(requested_k, len(FEATURE_POOL))
        selector, A_k, B_k, selected, _ = select_features_from_training(
            A, inner_tr.defects.values, B, FEATURE_POOL, k)
        clf = RandomForestClassifier(n_estimators=350, max_depth=None, min_samples_leaf=1,
                                     max_features="sqrt", class_weight=None,
                                     random_state=RANDOM_STATE, n_jobs=-1)
        clf.fit(A_k, inner_tr.defects)
        p = clf.predict_proba(B_k)[:,1]
        t, score = optimize_threshold(inner_va.defects.values, p)
        detail = validation_objective(inner_va.defects.values, p, t)[1]
        rows.append({"K":k, "Validation_Objective":score, "Validation_F1":detail["F1"],
                     "Validation_Accuracy":detail["Accuracy"], "Validation_Balanced_Accuracy":detail["Balanced_Accuracy"],
                     "Validation_MCC":detail["MCC"], "Threshold":t, "Features":selected})
    df = pd.DataFrame(rows).sort_values(
        ["Validation_Objective","Validation_F1","Validation_Accuracy"], ascending=False)
    return int(df.iloc[0].K), df


## 7. Feature Selection

In [ ]:
print("Requested feature-count candidates:", FEATURE_COUNTS)
print("Effective feature-count candidates:", EFFECTIVE_FEATURE_COUNTS)
print("Mutual-information selection is fitted inside each LODO training fold with a fixed random state.")


## 8. Baseline Machine Learning Models

In [ ]:
# Expanded but bounded search spaces. Class/positive weighting is tuned instead of fixed to "balanced".
RF_SPACE = {
    "n_estimators":[300,500,700], "max_depth":[None,8,12,18,24],
    "min_samples_split":[2,4,8,12], "min_samples_leaf":[1,2,3,5],
    "max_features":["sqrt","log2",0.6,0.8], "weight_factor":[0.45,0.60,0.75,0.90,1.0,1.15,1.30,1.5]
}
XGB_SPACE = {
    "n_estimators":[300,500,700], "max_depth":[2,3,4,5,6],
    "learning_rate":[0.02,0.04,0.06,0.10], "subsample":[0.75,0.85,0.95,1.0],
    "colsample_bytree":[0.7,0.85,1.0], "min_child_weight":[1,3,5,8],
    "gamma":[0,0.05,0.15,0.3], "reg_lambda":[0.5,1,3,5,10],
    "reg_alpha":[0,0.01,0.1,0.3], "weight_factor":[0.40,0.55,0.70,0.85,1.0,1.15,1.30,1.5]
}
LGB_SPACE = {
    "n_estimators":[300,500,700], "learning_rate":[0.02,0.04,0.06,0.10],
    "num_leaves":[7,15,23,31,47], "max_depth":[-1,5,8,12],
    "min_child_samples":[10,15,25,40], "subsample":[0.75,0.85,0.95,1.0],
    "colsample_bytree":[0.7,0.85,1.0], "reg_lambda":[0,1,2,5,10],
    "reg_alpha":[0,0.01,0.1,0.3], "weight_factor":[0.40,0.55,0.70,0.85,1.0,1.15,1.30,1.5]
}
CAT_SPACE = {
    "iterations":[400,600,800], "depth":[4,5,6,7,8], "learning_rate":[0.02,0.04,0.06,0.10],
    "l2_leaf_reg":[1,3,5,7,10], "random_strength":[0.1,0.5,1.0,1.5,2.0],
    "border_count":[64,128,254], "weight_factor":[0.40,0.55,0.70,0.85,1.0,1.15,1.30,1.5]
}

def _make_tuned_tree(name, p, scale_pos_weight):
    wf = float(p.pop("weight_factor", 1.0))
    if name == "Random Forest":
        return RandomForestClassifier(**p, class_weight={0:1.0,1:wf}, random_state=RANDOM_STATE, n_jobs=-1)
    if name == "XGBoost":
        return XGBClassifier(**p, objective="binary:logistic", eval_metric="logloss",
                              scale_pos_weight=float(scale_pos_weight)*wf, random_state=RANDOM_STATE,
                              n_jobs=-1, tree_method="hist")
    if name == "LightGBM":
        return LGBMClassifier(**p, class_weight={0:1.0,1:wf}, random_state=RANDOM_STATE,
                              verbose=-1, n_jobs=-1)
    if name == "CatBoost":
        return CatBoostClassifier(**p, loss_function="Logloss", class_weights=[1.0,wf*float(scale_pos_weight)],
                                  random_seed=RANDOM_STATE, verbose=False, allow_writing_files=False)
    raise ValueError(name)

def tune_tree_model(name, Xtr, ytr, Xva, yva, scale_pos_weight):
    spaces = {"Random Forest":RF_SPACE, "XGBoost":XGB_SPACE, "LightGBM":LGB_SPACE, "CatBoost":CAT_SPACE}
    candidates = list(ParameterSampler(spaces[name], n_iter=TREE_N_ITER, random_state=RANDOM_STATE))
    best = None
    for p0 in candidates:
        p = p0.copy()
        model = _make_tuned_tree(name, p, scale_pos_weight)
        if name == "XGBoost":
            model.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
            if getattr(model, "best_iteration", None) is not None and model.best_iteration is not None:
                p["n_estimators"] = int(model.best_iteration) + 1
        elif name == "LightGBM":
            model.fit(Xtr, ytr, eval_set=[(Xva, yva)], callbacks=[early_stopping(50, verbose=False)])
            if getattr(model, "best_iteration_", None): p["n_estimators"] = int(model.best_iteration_)
        elif name == "CatBoost":
            model.fit(Xtr, ytr, eval_set=(Xva, yva), early_stopping_rounds=50, verbose=False)
            if model.get_best_iteration() >= 0: p["iterations"] = int(model.get_best_iteration()) + 1
        else:
            model.fit(Xtr, ytr)
        pv = model.predict_proba(Xva)[:,1]
        t, score = optimize_threshold(yva, pv)
        detail = validation_objective(yva, pv, t)[1]
        if best is None or (score,detail["F1"],detail["Accuracy"]) > (best["objective"],best["f1"],best["accuracy"]):
            best = {"params":p.copy(), "objective":score, "f1":detail["F1"],
                    "accuracy":detail["Accuracy"], "balanced_accuracy":detail["Balanced_Accuracy"],
                    "mcc":detail["MCC"], "threshold":t}
    return best

def make_final_tree_model(name, params, scale_pos_weight):
    p = params.copy()
    return _make_tuned_tree(name, p, scale_pos_weight)


## 9. Optimized ANFIS

In [ ]:
class SugenoANFIS(nn.Module):
    # First-order Sugeno ANFIS with Gaussian membership functions.
    # The LODO pipeline limits ANFIS to a small number of inputs so that
    # the Cartesian rule base remains computationally controlled.
    def __init__(self, n_inputs, n_mfs=2, lr=0.003, epochs=100, weight_decay=1e-4,
                 sigma_scale=0.65, random_state=42, patience=15, min_delta=1e-5):
        super().__init__()
        self.n_inputs, self.n_mfs = int(n_inputs), int(n_mfs)
        self.lr, self.epochs, self.weight_decay = float(lr), int(epochs), float(weight_decay)
        self.random_state, self.patience, self.min_delta = int(random_state), int(patience), float(min_delta)
        self.sigma_scale = float(sigma_scale)
        self.rules = list(itertools.product(range(self.n_mfs), repeat=self.n_inputs))
        self.n_rules = len(self.rules)
        if self.n_rules > 81:
            raise ValueError(
                f"ANFIS rule base has {self.n_rules} rules; maximum allowed is 81. "
                f"Reduce ANFIS inputs or membership functions."
            )
        self.register_buffer("rule_indices", torch.tensor(self.rules, dtype=torch.long))
        self.centers = nn.Parameter(torch.zeros(self.n_inputs, self.n_mfs))
        self.log_sigmas = nn.Parameter(torch.zeros(self.n_inputs, self.n_mfs))
        self.consequents = nn.Parameter(torch.zeros(self.n_rules, self.n_inputs + 1))
        self.to(DEVICE)

    def _initialize_premises(self, X):
        X = np.asarray(X, dtype=np.float32)
        qs = np.array([0.5]) if self.n_mfs == 1 else np.linspace(0.15,0.85,self.n_mfs)
        with torch.no_grad():
            for j in range(self.n_inputs):
                c = np.quantile(X[:,j], qs).astype(np.float32)
                s = max(float(np.std(X[:,j]))*self.sigma_scale, 0.25)
                self.centers[j].copy_(torch.tensor(c, device=DEVICE))
                self.log_sigmas[j].fill_(np.log(s))

    def _membership(self, X):
        sigma = torch.exp(self.log_sigmas).clamp_min(1e-4)
        return torch.exp(-0.5*((X.unsqueeze(2)-self.centers.unsqueeze(0))/sigma.unsqueeze(0))**2).clamp_min(1e-8)

    def _firing_strengths(self, X):
        mu = self._membership(X)
        idx = self.rule_indices.T.unsqueeze(0).expand(X.shape[0],-1,-1)
        selected = torch.gather(mu,2,idx)
        return torch.softmax(torch.log(selected.clamp_min(1e-8)).sum(dim=1), dim=1)

    def forward(self, X):
        weights = self._firing_strengths(X)
        X_aug = torch.cat([torch.ones(X.shape[0],1,device=X.device,dtype=X.dtype),X],dim=1)
        rule_outputs = X_aug @ self.consequents.T
        return (weights*rule_outputs).sum(dim=1)

    def fit(self, X, y, X_val, y_val):
        set_global_seed(self.random_state)
        X, y = np.asarray(X,np.float32), np.asarray(y,np.float32)
        X_val, y_val = np.asarray(X_val,np.float32), np.asarray(y_val,np.float32)
        self._initialize_premises(X)
        Xt, yt = torch.tensor(X,device=DEVICE), torch.tensor(y,device=DEVICE)
        Xv, yv = torch.tensor(X_val,device=DEVICE), torch.tensor(y_val,device=DEVICE)
        counts = np.bincount(y.astype(int),minlength=2).astype(float)
        cw = np.array([len(y)/(2*max(counts[0],1)),len(y)/(2*max(counts[1],1))],dtype=np.float32)
        sw = torch.tensor(cw,device=DEVICE)[yt.long()]
        opt = torch.optim.AdamW(self.parameters(),lr=self.lr,weight_decay=self.weight_decay)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="min",factor=0.5,patience=6,min_lr=1e-5)
        best, state, wait = np.inf, None, 0
        self.history_ = []
        for epoch in range(self.epochs):
            self.train(); opt.zero_grad(set_to_none=True)
            loss = nn.functional.binary_cross_entropy_with_logits(self(Xt),yt,weight=sw)
            loss.backward(); torch.nn.utils.clip_grad_norm_(self.parameters(),5.0); opt.step()
            self.eval()
            with torch.no_grad():
                vl = nn.functional.binary_cross_entropy_with_logits(self(Xv),yv)
            val_loss = float(vl.item()); sched.step(val_loss)
            self.history_.append((float(loss.item()),val_loss))
            if val_loss < best-self.min_delta:
                best, wait = val_loss, 0
                state = {k:v.detach().cpu().clone() for k,v in self.state_dict().items()}
            else:
                wait += 1
                if wait >= self.patience: break
        if state is not None: self.load_state_dict(state)
        self.eval()
        with torch.no_grad(): p = torch.sigmoid(self(Xv)).cpu().numpy()
        self.best_threshold_, self.best_val_f1_ = optimize_threshold(y_val,p)
        return self

    def predict_proba_defect(self,X):
        self.eval()
        Xt = torch.tensor(np.asarray(X,np.float32),device=DEVICE)
        with torch.no_grad(): return torch.sigmoid(self(Xt)).cpu().numpy()


def prepare_anfis_inputs(Xtr, ytr, Xva, max_inputs=ANFIS_MAX_INPUTS):
    Xtr = np.asarray(Xtr); Xva = np.asarray(Xva)
    k = min(int(max_inputs), Xtr.shape[1])
    selector = SelectKBest(score_func=lambda X,y: mutual_info_classif(X,y,random_state=RANDOM_STATE), k=k)
    A = selector.fit_transform(Xtr, np.asarray(ytr))
    B = selector.transform(Xva)
    return selector, A, B

def tune_anfis(Xtr,ytr,Xva,yva):
    # Search both dimensionality and ANFIS hyperparameters while respecting <=81 rules.
    candidates = []
    for n_inputs in range(2, min(ANFIS_MAX_INPUTS, Xtr.shape[1]) + 1):
        selector, A, B = prepare_anfis_inputs(Xtr, ytr, Xva, n_inputs)
        for n_mfs in [2,3]:
            if n_mfs ** n_inputs > 81: continue
            for lr in [0.0008,0.0015,0.003,0.006]:
                for sigma_scale in [0.45,0.60,0.75,0.90]:
                    candidates.append((n_inputs,n_mfs,lr,sigma_scale,selector,A,B))
    # Deterministic subset to keep Colab runtime practical while being much broader than v1.
    rng = np.random.default_rng(RANDOM_STATE)
    if len(candidates) > 36:
        chosen = rng.choice(len(candidates), size=36, replace=False)
        candidates = [candidates[int(i)] for i in chosen]
    best = None
    for n_inputs,n_mfs,lr,sigma_scale,selector,A,B in candidates:
        model = SugenoANFIS(A.shape[1], n_mfs=n_mfs, lr=lr, epochs=180,
                            sigma_scale=sigma_scale, random_state=RANDOM_STATE, patience=25)
        model.fit(A,ytr,B,yva)
        p = model.predict_proba_defect(B)
        t, score = optimize_threshold(yva,p)
        detail = validation_objective(yva,p,t)[1]
        row = {"n_inputs":n_inputs,"n_mfs":n_mfs,"lr":lr,"sigma_scale":sigma_scale,
               "objective":score,"f1":detail["F1"],"accuracy":detail["Accuracy"],
               "balanced_accuracy":detail["Balanced_Accuracy"],"mcc":detail["MCC"],"threshold":t}
        if best is None or (score,detail["F1"],detail["Accuracy"]) > (best["objective"],best["f1"],best["accuracy"]):
            best = {"params":{"n_mfs":n_mfs,"lr":lr,"sigma_scale":sigma_scale,"n_inputs":n_inputs},
                    "objective":score,"f1":detail["F1"],"accuracy":detail["Accuracy"],
                    "balanced_accuracy":detail["Balanced_Accuracy"],"mcc":detail["MCC"],"threshold":t}
    if best is None: raise RuntimeError("No valid ANFIS configuration was found.")
    return best


## 10. Hybrid ML + ANFIS Ensemble

In [ ]:
EQUAL_WEIGHTS = {name:0.20 for name in MODEL_NAMES[:5]}

def optimize_ensemble_weights(y_val, prob_dict):
    names = MODEL_NAMES[:5]
    # Coarse-to-fine simplex search. Coarse grid first, then local refinements.
    best_w = np.ones(len(names))/len(names)
    best_score = -np.inf
    def evaluate(w):
        p = sum(float(wi)*prob_dict[n] for wi,n in zip(w,names))
        t, score = optimize_threshold(y_val,p)
        detail = validation_objective(y_val,p,t)[1]
        return score, detail, t
    grid = np.arange(0.0,1.001,0.1)
    for w4 in grid:
        for w3 in grid:
            for w2 in grid:
                for w1 in grid:
                    w0 = 1.0-w1-w2-w3-w4
                    if w0 < -1e-9: continue
                    w=np.array([w0,w1,w2,w3,w4])
                    score,detail,t=evaluate(w)
                    if (score,detail["Accuracy"],detail["F1"],detail["Balanced_Accuracy"]) > (best_score,-1,-1,-1):
                        best_score=score; best_w=w.copy(); best_t=t
    # Local fine search around the coarse winner.
    step=0.025
    center=best_w.copy()
    for _ in range(2):
        candidates=[]
        for i in range(len(names)):
            for delta in [-2,-1,0,1,2]:
                w=center.copy(); w[i]=np.clip(w[i]+delta*step,0,1)
                s=w.sum()
                if s<=0: continue
                w=w/s
                candidates.append(w)
        for w in candidates:
            score,detail,t=evaluate(w)
            if (score,detail["Accuracy"],detail["F1"],detail["Balanced_Accuracy"]) > (best_score,-1,-1,-1):
                best_score=score; best_w=w.copy(); best_t=t
        center=best_w.copy()
    return dict(zip(names,best_w)), best_score

def hybrid_probability(prob_dict, weights):
    return sum(weights[n]*prob_dict[n] for n in MODEL_NAMES[:5])

def fit_meta_learner(y_val, prob_dict):
    names = MODEL_NAMES[:5]
    Z = np.column_stack([prob_dict[n] for n in names])
    # Small regularization grid; the meta learner sees only internal-validation predictions.
    best=None
    for C in [0.01,0.03,0.1,0.3,1,3,10,30]:
        meta=LogisticRegression(C=C,class_weight=None,solver="lbfgs",max_iter=2000,random_state=RANDOM_STATE)
        meta.fit(Z,y_val)
        p=meta.predict_proba(Z)[:,1]
        t,score=optimize_threshold(y_val,p)
        detail=validation_objective(y_val,p,t)[1]
        cand=(score,detail["F1"],detail["Accuracy"])
        if best is None or cand>(best["score"],best["f1"],best["accuracy"]):
            best={"model":meta,"score":score,"f1":detail["F1"],"accuracy":detail["Accuracy"],"threshold":t}
    return best


In [ ]:
# ============================================================
# REQUIRED METRICS FUNCTION — FIXED
# This cell MUST run before Section 11 (LODO).
# ============================================================

def compute_metrics(y_true, proba, threshold=0.50):
    """Return the complete metric set used throughout the notebook."""
    y_true = np.asarray(y_true).astype(int)
    proba = np.asarray(proba).astype(float)
    pred = (proba >= float(threshold)).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, pred, labels=[0, 1]
    ).ravel()

    accuracy = accuracy_score(y_true, pred)
    precision = precision_score(y_true, pred, zero_division=0)
    recall = recall_score(y_true, pred, zero_division=0)
    f1 = f1_score(y_true, pred, zero_division=0)
    balanced_accuracy = balanced_accuracy_score(y_true, pred)
    specificity = tn / max(tn + fp, 1)
    mcc = matthews_corrcoef(y_true, pred)

    if len(np.unique(y_true)) >= 2:
        roc_auc = roc_auc_score(y_true, proba)
        pr_auc = average_precision_score(y_true, proba)
    else:
        roc_auc = np.nan
        pr_auc = np.nan

    mae = mean_absolute_error(y_true, proba)
    mse = mean_squared_error(y_true, proba)

    try:
        r2 = r2_score(y_true, proba)
    except Exception:
        r2 = np.nan

    return {
        "Accuracy": float(accuracy),
        "Precision": float(precision),
        "Recall": float(recall),
        "F1": float(f1),
        "ROC_AUC": float(roc_auc) if not np.isnan(roc_auc) else np.nan,
        "PR_AUC": float(pr_auc) if not np.isnan(pr_auc) else np.nan,
        "Balanced_Accuracy": float(balanced_accuracy),
        "Specificity": float(specificity),
        "MCC": float(mcc),
        "MAE": float(mae),
        "MSE": float(mse),
        "R2": float(r2) if not np.isnan(r2) else np.nan,
    }

print("PASS — compute_metrics() is defined and ready for LODO.")


## 11. Leave-One-Dataset-Out Evaluation

In [ ]:
## 11. Leave-One-Dataset-Out Evaluation — FAST Accuracy-Optimized

# This cell is a speed-focused replacement for the original LODO cell.

# HARD SAFETY CHECKS — prevents silent creation of an incomplete predictions object.
if "compute_metrics" not in globals():
    raise RuntimeError("compute_metrics() is missing. Run the required metrics cell immediately before Section 11.")
if "MODEL_NAMES" not in globals() or len(MODEL_NAMES) != 6:
    raise RuntimeError("MODEL_NAMES must contain exactly six models before Section 11.")
if "SELECTED_DATASETS" not in globals() or len(SELECTED_DATASETS) != 5:
    raise RuntimeError("SELECTED_DATASETS must contain the five LODO projects before Section 11.")

# It preserves strict LODO: the held-out project is NEVER used for tuning.
# Main speedups: fewer tree/ANFIS candidates, vectorized threshold search,
# lightweight ensemble search, and no logistic-stacking fit inside each fold.

FAST_TREE_N_ITER = 6
FAST_ANFIS_CONFIGS = 6
FAST_ANFIS_EPOCHS = 80
FAST_ANFIS_PATIENCE = 12
FAST_WEIGHT_SAMPLES = 160
FAST_THRESHOLD_GRID = np.arange(0.02, 0.981, 0.02)


def fast_validation_objective(y_true, proba, threshold):
    y_true = np.asarray(y_true).astype(int)
    pred = (np.asarray(proba) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0,1]).ravel()
    acc = (tp + tn) / max(tp + tn + fp + fn, 1)
    rec = tp / max(tp + fn, 1)
    prec = tp / max(tp + fp, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-12)
    spec = tn / max(tn + fp, 1)
    bal = 0.5 * (rec + spec)
    mcc = matthews_corrcoef(y_true, pred)
    score = (OBJECTIVE_WEIGHTS["Accuracy"] * acc +
             OBJECTIVE_WEIGHTS["F1"] * f1 +
             OBJECTIVE_WEIGHTS["Balanced_Accuracy"] * bal +
             OBJECTIVE_WEIGHTS["MCC_scaled"] * ((mcc + 1.0) / 2.0))
    return score, {"Accuracy":acc, "F1":f1, "Balanced_Accuracy":bal, "MCC":mcc}


def fast_optimize_threshold(y_true, proba):
    """Vectorized threshold search; avoids calling sklearn metrics ~50-100 times."""
    y = np.asarray(y_true).astype(int)
    p = np.asarray(proba).astype(float)
    pred = p[:, None] >= FAST_THRESHOLD_GRID[None, :]
    yy = y[:, None]
    tp = np.sum((pred == 1) & (yy == 1), axis=0)
    tn = np.sum((pred == 0) & (yy == 0), axis=0)
    fp = np.sum((pred == 1) & (yy == 0), axis=0)
    fn = np.sum((pred == 0) & (yy == 1), axis=0)
    acc = (tp + tn) / max(len(y), 1)
    rec = tp / np.maximum(tp + fn, 1)
    prec = tp / np.maximum(tp + fp, 1)
    f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-12)
    spec = tn / np.maximum(tn + fp, 1)
    bal = 0.5 * (rec + spec)
    denom = np.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))
    mcc = np.divide(tp*tn-fp*fn, denom, out=np.zeros_like(acc, dtype=float), where=denom>0)
    score = (OBJECTIVE_WEIGHTS["Accuracy"] * acc +
             OBJECTIVE_WEIGHTS["F1"] * f1 +
             OBJECTIVE_WEIGHTS["Balanced_Accuracy"] * bal +
             OBJECTIVE_WEIGHTS["MCC_scaled"] * ((mcc + 1.0) / 2.0))
    # Accuracy is the first tie-breaker, then F1, balanced accuracy and MCC.
    best = max(range(len(FAST_THRESHOLD_GRID)),
               key=lambda i:(score[i],acc[i],f1[i],bal[i],mcc[i]))
    detail = {"Accuracy":float(acc[best]), "F1":float(f1[best]),
              "Balanced_Accuracy":float(bal[best]), "MCC":float(mcc[best])}
    return float(FAST_THRESHOLD_GRID[best]), float(score[best]), detail


def fast_tune_tree_model(name, Xtr, ytr, Xva, yva, scale_pos_weight):
    spaces = {"Random Forest":RF_SPACE, "XGBoost":XGB_SPACE,
              "LightGBM":LGB_SPACE, "CatBoost":CAT_SPACE}
    candidates = list(ParameterSampler(spaces[name], n_iter=FAST_TREE_N_ITER,
                                       random_state=RANDOM_STATE))
    best = None
    for p0 in candidates:
        p = p0.copy()
        model = _make_tuned_tree(name, p, scale_pos_weight)
        if name == "XGBoost":
            model.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
            if getattr(model, "best_iteration", None) is not None:
                p["n_estimators"] = int(model.best_iteration) + 1
        elif name == "LightGBM":
            model.fit(Xtr, ytr, eval_set=[(Xva, yva)], callbacks=[early_stopping(30, verbose=False)])
            if getattr(model, "best_iteration_", None):
                p["n_estimators"] = int(model.best_iteration_)
        elif name == "CatBoost":
            model.fit(Xtr, ytr, eval_set=(Xva, yva), early_stopping_rounds=30, verbose=False)
            if model.get_best_iteration() >= 0:
                p["iterations"] = int(model.get_best_iteration()) + 1
        else:
            model.fit(Xtr, ytr)
        pv = model.predict_proba(Xva)[:,1]
        t, score, detail = fast_optimize_threshold(yva, pv)
        cand = (score, detail["Accuracy"], detail["F1"], detail["Balanced_Accuracy"])
        if best is None or cand > (best["objective"],best["accuracy"],best["f1"],best["balanced_accuracy"]):
            best = {"params":p.copy(), "objective":score, "f1":detail["F1"],
                    "accuracy":detail["Accuracy"], "balanced_accuracy":detail["Balanced_Accuracy"],
                    "mcc":detail["MCC"], "threshold":t}
    return best


def fast_tune_anfis(Xtr,ytr,Xva,yva):
    # Deterministic small search. All configurations obey the <=81-rule constraint.
    dims = min(ANFIS_MAX_INPUTS, Xtr.shape[1])
    base = [(2,2,0.0015,0.60),(2,2,0.0030,0.75),(3,2,0.0015,0.60),
            (3,2,0.0030,0.75),(4,2,0.0015,0.60),(4,2,0.0030,0.75)]
    configs = [c for c in base if c[0] <= dims and c[1]**c[0] <= 81][:FAST_ANFIS_CONFIGS]
    best = None
    for n_inputs,n_mfs,lr,sigma_scale in configs:
        _,A,B = prepare_anfis_inputs(Xtr,ytr,Xva,n_inputs)
        model = SugenoANFIS(A.shape[1], n_mfs=n_mfs, lr=lr,
                            epochs=FAST_ANFIS_EPOCHS, sigma_scale=sigma_scale,
                            random_state=RANDOM_STATE, patience=FAST_ANFIS_PATIENCE)
        model.fit(A,ytr,B,yva)
        p = model.predict_proba_defect(B)
        t, score, detail = fast_optimize_threshold(yva,p)
        cand = (score, detail["Accuracy"], detail["F1"], detail["Balanced_Accuracy"])
        if best is None or cand > (best["objective"],best["accuracy"],best["f1"],best["balanced_accuracy"]):
            best = {"params":{"n_mfs":n_mfs,"lr":lr,"sigma_scale":sigma_scale,"n_inputs":n_inputs},
                    "objective":score,"f1":detail["F1"],"accuracy":detail["Accuracy"],
                    "balanced_accuracy":detail["Balanced_Accuracy"],"mcc":detail["MCC"],"threshold":t}
    if best is None:
        raise RuntimeError("No valid fast ANFIS configuration was found.")
    return best


def fast_ensemble_search(y_val, prob_dict):
    names = MODEL_NAMES[:5]
    P = np.column_stack([prob_dict[n] for n in names])
    rng = np.random.default_rng(RANDOM_STATE)
    # Include equal weights and one-hot candidates, then a small Dirichlet search.
    weights = [np.ones(5)/5]
    weights += [np.eye(5)[i] for i in range(5)]
    weights += list(rng.dirichlet(np.ones(5)*0.8, size=FAST_WEIGHT_SAMPLES))
    best = None
    for w in weights:
        p = P @ np.asarray(w)
        t, score, detail = fast_optimize_threshold(y_val,p)
        cand = (score,detail["Accuracy"],detail["F1"],detail["Balanced_Accuracy"])
        if best is None or cand > best[0]:
            best = (cand, np.asarray(w), t)
    return dict(zip(names,best[1])), float(best[0][0]), float(best[2])


all_results, predictions, model_objects, feature_importance_records, fold_metadata = [], {}, {}, [], []

for held_out in SELECTED_DATASETS:
    print("\n"+"="*70)
    print(f"FAST ACCURACY-OPTIMIZED LODO: Holding out {held_out}")
    train_projects=[d for d in SELECTED_DATASETS if d!=held_out]
    print("Training datasets:", ", ".join(train_projects))
    print("="*70)

    train_df=data[data.dataset.isin(train_projects)].reset_index(drop=True)
    test_df=data[data.dataset==held_out].reset_index(drop=True)

    feature_k, feature_tuning=choose_feature_count(train_df)
    inner_tr,inner_va=make_internal_split(train_df)
    imp_i,sc_i=SimpleImputer(strategy="median"),StandardScaler()
    A0=sc_i.fit_transform(imp_i.fit_transform(inner_tr[FEATURE_POOL]))
    B0=sc_i.transform(imp_i.transform(inner_va[FEATURE_POOL]))
    sel_i,A,B,selected_features,_=select_features_from_training(A0,inner_tr.defects.values,B0,FEATURE_POOL,feature_k)
    yA,yB=inner_tr.defects.values,inner_va.defects.values
    neg,pos=np.bincount(yA,minlength=2); spw=neg/max(pos,1)

    tuned={}
    for name in MODEL_NAMES[:4]:
        tuned[name]=fast_tune_tree_model(name,A,yA,B,yB,spw)
        print(f"{name}: objective={tuned[name]['objective']:.4f}, F1={tuned[name]['f1']:.4f}, Accuracy={tuned[name]['accuracy']:.4f}, threshold={tuned[name]['threshold']:.2f}")
    anfis_tuned=fast_tune_anfis(A,yA,B,yB)
    print(f"ANFIS: objective={anfis_tuned['objective']:.4f}, F1={anfis_tuned['f1']:.4f}, Accuracy={anfis_tuned['accuracy']:.4f}, threshold={anfis_tuned['threshold']:.2f}, inputs={anfis_tuned['params']['n_inputs']}, rules={anfis_tuned['params']['n_mfs']**anfis_tuned['params']['n_inputs']}")

    # Final preprocessing/feature selection is fitted ONLY on the four training projects.
    imp,sc=SimpleImputer(strategy="median"),StandardScaler()
    Xtr0=sc.fit_transform(imp.fit_transform(train_df[FEATURE_POOL]))
    Xte0=sc.transform(imp.transform(test_df[FEATURE_POOL]))
    selector=SelectKBest(score_func=lambda X,y: mutual_info_classif(X,y,random_state=RANDOM_STATE),k=min(feature_k,len(FEATURE_POOL)))
    Xtr=selector.fit_transform(Xtr0,train_df.defects); Xte=selector.transform(Xte0)
    selected_final=list(np.asarray(FEATURE_POOL)[selector.get_support()])
    y_train,y_test=train_df.defects.values,test_df.defects.values
    neg,pos=np.bincount(y_train,minlength=2); spw_final=neg/max(pos,1)

    # Final four tree models.
    for name in MODEL_NAMES[:4]:
        model=make_final_tree_model(name,tuned[name]["params"],spw_final)
        model.fit(Xtr,y_train)
        p=model.predict_proba(Xte)[:,1]
        threshold=tuned[name]["threshold"]
        predictions[(held_out,name)]={"y_true":y_test.copy(),"proba":p.copy(),"threshold":threshold}
        model_objects[(held_out,name)]=model
        all_results.append({"Dataset":held_out,"Model":name,"Threshold":threshold,**compute_metrics(y_test,p,threshold)})
        if hasattr(model,"feature_importances_"):
            for f,impv in zip(selected_final,model.feature_importances_):
                feature_importance_records.append({"Dataset":held_out,"Model":name,"Feature":f,"Importance":float(impv)})

    # Final ANFIS, trained only with the non-held-out projects.
    anfis_k=int(anfis_tuned["params"]["n_inputs"])
    anfis_selector=SelectKBest(score_func=lambda X,y: mutual_info_classif(X,y,random_state=RANDOM_STATE),k=min(anfis_k,Xtr.shape[1]))
    Xtr_anfis=anfis_selector.fit_transform(Xtr,y_train); Xte_anfis=anfis_selector.transform(Xte)
    idx_tr,idx_va=train_test_split(np.arange(len(y_train)),test_size=VALIDATION_SIZE,stratify=y_train,random_state=RANDOM_STATE)
    anfis=SugenoANFIS(Xtr_anfis.shape[1],n_mfs=anfis_tuned["params"]["n_mfs"],lr=anfis_tuned["params"]["lr"],sigma_scale=anfis_tuned["params"]["sigma_scale"],epochs=FAST_ANFIS_EPOCHS,random_state=RANDOM_STATE,patience=FAST_ANFIS_PATIENCE)
    anfis.fit(Xtr_anfis[idx_tr],y_train[idx_tr],Xtr_anfis[idx_va],y_train[idx_va])
    p_anfis=anfis.predict_proba_defect(Xte_anfis)
    predictions[(held_out,"ANFIS")]={"y_true":y_test.copy(),"proba":p_anfis.copy(),"threshold":anfis_tuned["threshold"]}
    model_objects[(held_out,"ANFIS")]=anfis
    all_results.append({"Dataset":held_out,"Model":"ANFIS","Threshold":anfis_tuned["threshold"],**compute_metrics(y_test,p_anfis,anfis_tuned["threshold"])})

    # Refit the five base models on the internal split to obtain validation probabilities.
    # These predictions are used only for frozen ensemble-weight/threshold selection.
    val_probs={}
    for name in MODEL_NAMES[:4]:
        m=make_final_tree_model(name,tuned[name]["params"],spw)
        m.fit(A,yA)
        val_probs[name]=m.predict_proba(B)[:,1]
    _,A_anfis,B_anfis=prepare_anfis_inputs(A,yA,B,anfis_tuned["params"]["n_inputs"])
    a_val=SugenoANFIS(A_anfis.shape[1],n_mfs=anfis_tuned["params"]["n_mfs"],lr=anfis_tuned["params"]["lr"],sigma_scale=anfis_tuned["params"]["sigma_scale"],epochs=FAST_ANFIS_EPOCHS,random_state=RANDOM_STATE,patience=FAST_ANFIS_PATIENCE)
    a_val.fit(A_anfis,yA,B_anfis,yB)
    val_probs["ANFIS"]=a_val.predict_proba_defect(B_anfis)

    # Fast accuracy-prioritized weighted hybrid. Logistic stacking is deliberately omitted here
    # because it is slower and can overfit the same internal-validation predictions it fits on.
    opt_weights,opt_score,hybrid_t=fast_ensemble_search(yB,val_probs)
    test_probs={name:predictions[(held_out,name)]["proba"] for name in MODEL_NAMES[:5]}
    p_hybrid=hybrid_probability(test_probs,opt_weights)
    predictions[(held_out,"Hybrid ML + ANFIS")]={"y_true":y_test.copy(),"proba":p_hybrid.copy(),"threshold":hybrid_t}
    model_objects[(held_out,"Hybrid ML + ANFIS")]={"type":"weighted_fast","weights":opt_weights}
    all_results.append({"Dataset":held_out,"Model":"Hybrid ML + ANFIS","Threshold":hybrid_t,**compute_metrics(y_test,p_hybrid,hybrid_t)})

    fold_metadata.append({"Dataset":held_out,"Feature_Count":feature_k,"Selected_Features":"|".join(selected_final),
                          "Hybrid_Selection":"weighted_fast","Hybrid_Validation_Objective":opt_score,
                          "Hybrid_Weights_or_Meta":json.dumps(opt_weights)})
    print("Chosen hybrid: weighted_fast | validation objective:",round(opt_score,4),"| threshold:",round(hybrid_t,2))

results_df = pd.DataFrame(all_results)
expected_keys = [(d, m) for d in SELECTED_DATASETS for m in MODEL_NAMES]
missing_keys = [k for k in expected_keys if k not in predictions]

print("\n" + "=" * 70)
print("LODO VALIDATION CHECK")
print("=" * 70)
print("Expected prediction objects:", len(expected_keys))
print("Actual prediction objects:", len(predictions))
print("Result rows:", len(results_df))

if missing_keys:
    print("Missing prediction keys:")
    for key in missing_keys:
        print("  ", key)
    raise RuntimeError("LODO did not create all 30 dataset/model predictions.")

if len(results_df) != 30:
    raise RuntimeError(f"Expected 30 LODO result rows, got {len(results_df)}.")

if set(results_df["Model"]) != set(MODEL_NAMES):
    raise RuntimeError("LODO results do not contain all six models.")

for key, item in predictions.items():
    if len(item["y_true"]) != len(item["proba"]):
        raise RuntimeError(f"Length mismatch in prediction object {key}.")
    if not np.isfinite(item["proba"]).all():
        raise RuntimeError(f"Non-finite probabilities found in {key}.")

display(results_df.sort_values(["Dataset", "F1"], ascending=[True, False]).round(4))
print("\nPASS — 30/30 prediction objects created.")
print("PASS — 5 datasets × 6 models completed.")
print("FAST LODO complete.")


## 12. Classification Metrics

In [ ]:
LODO_summary=results_df.groupby("Model").agg(
    Accuracy_mean=("Accuracy","mean"),Accuracy_std=("Accuracy","std"),
    Precision_mean=("Precision","mean"),Precision_std=("Precision","std"),
    Recall_mean=("Recall","mean"),Recall_std=("Recall","std"),
    F1_mean=("F1","mean"),F1_std=("F1","std"),
    ROC_AUC_mean=("ROC_AUC","mean"),ROC_AUC_std=("ROC_AUC","std"),
    PR_AUC_mean=("PR_AUC","mean"),PR_AUC_std=("PR_AUC","std"),
    Balanced_Accuracy_mean=("Balanced_Accuracy","mean"),Balanced_Accuracy_std=("Balanced_Accuracy","std"),
    Specificity_mean=("Specificity","mean"),Specificity_std=("Specificity","std"),
    MCC_mean=("MCC","mean"),MCC_std=("MCC","std")
).reset_index()
display(LODO_summary.sort_values(["F1_mean","Accuracy_mean"],ascending=False).round(4))
LODO_summary.to_csv(RESULTS_DIR/"tables"/"LODO_summary_enhanced.csv",index=False)


## 13. MAE / MSE / R²

MAE, MSE and R² use the defective-class probability. They are supplementary probability-based error measures, not conventional regression targets.

In [ ]:
LODO_MAE_MSE_R2 = results_df.groupby("Model").agg(
    MAE_mean=("MAE","mean"),MAE_std=("MAE","std"),
    MSE_mean=("MSE","mean"),MSE_std=("MSE","std"),
    R2_mean=("R2","mean"),R2_std=("R2","std")
).reset_index()
display(LODO_MAE_MSE_R2.round(4))

## 14. Actual vs Predicted Analysis

In [ ]:
for (d,m),item in predictions.items():
    y,p,t = item["y_true"],item["proba"],item["threshold"]
    pred = (p>=t).astype(int); x=np.arange(len(y))
    fig,ax=plt.subplots(figsize=(10,5))
    ax.plot(x,y,label="Actual Class",linewidth=1); ax.plot(x,p,label="Predicted Probability",linewidth=1)
    ax.set(xlabel="Sample Index",ylabel="Actual Class / Predicted Probability",
           title=f"Actual vs Predicted Probability — {d} — {m}"); ax.legend(); ax.grid(True,alpha=.25)
    fig.tight_layout(); fig.savefig(RESULTS_DIR/"plots"/f"{d}_{m.replace(' ','_')}_probability.png",dpi=220); plt.close(fig)
    fig,ax=plt.subplots(figsize=(10,5))
    ax.plot(x,y,label="Actual Class",linewidth=1); ax.plot(x,pred,label="Predicted Class",linewidth=1)
    ax.set(xlabel="Sample Index",ylabel="Actual Class / Predicted Class",
           title=f"Actual vs Predicted Class — {d} — {m}"); ax.set_yticks([0,1]); ax.legend(); ax.grid(True,alpha=.25)
    fig.tight_layout(); fig.savefig(RESULTS_DIR/"plots"/f"{d}_{m.replace(' ','_')}_class.png",dpi=220); plt.close(fig)
print("Actual-vs-predicted plots saved.")

## 15. Confusion Matrices

In [ ]:
for (d,m),item in predictions.items():
    y,p,t=item["y_true"],item["proba"],item["threshold"]; pred=(p>=t).astype(int)
    for norm,suffix in [(None,"raw"),("true","normalized")]:
        cm=confusion_matrix(y,pred,labels=[0,1],normalize=norm)
        fig,ax=plt.subplots(figsize=(5.5,4.5))
        sns.heatmap(cm,annot=True,fmt=".2f" if norm else "d",cmap="Blues",
                    xticklabels=["Non-defective","Defective"],yticklabels=["Non-defective","Defective"],ax=ax)
        ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(f"Confusion Matrix — {d} — {m} — {suffix}")
        fig.tight_layout(); fig.savefig(RESULTS_DIR/"confusion_matrices"/f"{d}_{m.replace(' ','_')}_{suffix}.png",dpi=220); plt.close(fig)
for m in MODEL_NAMES:
    yy=np.concatenate([predictions[(d,m)]["y_true"] for d in SELECTED_DATASETS])
    pp=np.concatenate([predictions[(d,m)]["proba"] for d in SELECTED_DATASETS])
    tt=np.mean([predictions[(d,m)]["threshold"] for d in SELECTED_DATASETS])
    pred=(pp>=tt).astype(int)
    cm=confusion_matrix(yy,pred,labels=[0,1])
    fig,ax=plt.subplots(figsize=(5.5,4.5))
    sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",
                xticklabels=["Non-defective","Defective"],yticklabels=["Non-defective","Defective"],ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(f"Pooled LODO Confusion Matrix — {m}")
    fig.tight_layout(); fig.savefig(RESULTS_DIR/"confusion_matrices"/f"pooled_{m.replace(' ','_')}_raw.png",dpi=220); plt.close(fig)


## 16. ROC Curves

In [ ]:
for d in SELECTED_DATASETS:
    fig,ax=plt.subplots(figsize=(8,6))
    for m in MODEL_NAMES:
        item=predictions[(d,m)]; fpr,tpr,_=roc_curve(item["y_true"],item["proba"])
        auc=roc_auc_score(item["y_true"],item["proba"]); ax.plot(fpr,tpr,label=f"{m} (AUC={auc:.3f})")
    ax.plot([0,1],[0,1],"--",linewidth=1); ax.set(xlabel="False Positive Rate",ylabel="True Positive Rate",
        title=f"ROC Comparison — {d}"); ax.legend(fontsize=8); fig.tight_layout()
    fig.savefig(RESULTS_DIR/"roc_curves"/f"{d}_roc_comparison.png",dpi=240); plt.close(fig)
# Pooled out-of-project predictions: each row is still a held-out LODO observation.
fig,ax=plt.subplots(figsize=(8,6))
for m in MODEL_NAMES:
    yy=np.concatenate([predictions[(d,m)]["y_true"] for d in SELECTED_DATASETS])
    pp=np.concatenate([predictions[(d,m)]["proba"] for d in SELECTED_DATASETS])
    fpr,tpr,_=roc_curve(yy,pp)
    ax.plot(fpr,tpr,label=f"{m} (AUC={roc_auc_score(yy,pp):.3f})")
ax.plot([0,1],[0,1],"--",linewidth=1)
ax.set(xlabel="False Positive Rate",ylabel="True Positive Rate",title="Pooled LODO ROC Comparison")
ax.legend(fontsize=8); fig.tight_layout()
fig.savefig(RESULTS_DIR/"roc_curves"/"pooled_lodo_roc_comparison.png",dpi=240); plt.close(fig)


## 17. Precision-Recall Curves

In [ ]:
# ============================================================
# 17. PRECISION-RECALL CURVES — V7 FIX
# ============================================================
# This section explicitly uses precision_recall_curve for both
# per-project and pooled LODO PR analysis.

for d in SELECTED_DATASETS:
    fig, ax = plt.subplots(figsize=(8,6))
    for m in MODEL_NAMES:
        item = predictions[(d,m)]
        precision, recall, _ = precision_recall_curve(item["y_true"], item["proba"])
        ap = average_precision_score(item["y_true"], item["proba"])
        ax.plot(recall, precision, label=f"{m} (AP={ap:.3f})")
    ax.set(
        xlabel="Recall",
        ylabel="Precision",
        title=f"Precision-Recall Comparison — {d}"
    )
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(
        RESULTS_DIR/"pr_curves"/f"{d}_pr_comparison.png",
        dpi=240
    )
    plt.close(fig)

# Pooled out-of-project predictions.
fig, ax = plt.subplots(figsize=(8,6))
for m in MODEL_NAMES:
    yy = np.concatenate([predictions[(d,m)]["y_true"] for d in SELECTED_DATASETS])
    pp = np.concatenate([predictions[(d,m)]["proba"] for d in SELECTED_DATASETS])
    precision, recall, _ = precision_recall_curve(yy, pp)
    ap = average_precision_score(yy, pp)
    ax.plot(recall, precision, label=f"{m} (AP={ap:.3f})")

ax.set(
    xlabel="Recall",
    ylabel="Precision",
    title="Pooled LODO Precision-Recall Comparison"
)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(
    RESULTS_DIR/"pr_curves"/"pooled_lodo_pr_comparison.png",
    dpi=240
)
plt.close(fig)

print("PASS — Precision-Recall curves generated correctly.")


## 18. Feature Importance

In [ ]:
fi = pd.DataFrame(feature_importance_records)
if not fi.empty:
    fi.to_csv(RESULTS_DIR/"feature_importance"/"feature_importance_by_lodo.csv",index=False)
    consolidated_fi=fi.groupby(["Model","Feature"])["Importance"].agg(["mean","std"]).reset_index()
    consolidated_fi.to_csv(RESULTS_DIR/"tables"/"feature_importance_consolidated.csv",index=False)
    top=consolidated_fi.groupby("Feature")["mean"].mean().sort_values(ascending=False).head(15)
    fig,ax=plt.subplots(figsize=(9,6)); top.sort_values().plot(kind="barh",ax=ax)
    ax.set(xlabel="Mean Importance",ylabel="Feature",title="Consolidated Tree-Model Feature Importance")
    fig.tight_layout(); fig.savefig(RESULTS_DIR/"feature_importance"/"top_features.png",dpi=240); plt.close(fig)

## 19. Ablation Study

In [ ]:
# ============================================================
# 19. ABLATION STUDY — V7
# ============================================================
ablation_rows = []

for d in SELECTED_DATASETS:
    # Individual models
    for m in MODEL_NAMES[:5]:
        r = results_df[(results_df.Dataset == d) & (results_df.Model == m)].iloc[0]
        ablation_rows.append({
            "Dataset": d,
            "Configuration": m,
            "Accuracy": r.Accuracy,
            "Balanced_Accuracy": r.Balanced_Accuracy,
            "Precision": r.Precision,
            "Recall": r.Recall,
            "F1": r.F1,
            "ROC_AUC": r.ROC_AUC,
            "PR_AUC": r.PR_AUC,
            "MCC": r.MCC
        })

    # ML-only ensemble: equal-weight average of the four tree models.
    y = predictions[(d, "Random Forest")]["y_true"]
    ml_p = np.mean(
        np.column_stack([
            predictions[(d, m)]["proba"] for m in MODEL_NAMES[:4]
        ]),
        axis=1
    )
    t, _ = optimize_threshold(y, ml_p)
    mm = compute_metrics(y, ml_p, t)

    ablation_rows.append({
        "Dataset": d,
        "Configuration": "ML ensemble without ANFIS",
        "Accuracy": mm["Accuracy"],
        "Balanced_Accuracy": mm["Balanced_Accuracy"],
        "Precision": mm["Precision"],
        "Recall": mm["Recall"],
        "F1": mm["F1"],
        "ROC_AUC": mm["ROC_AUC"],
        "PR_AUC": mm["PR_AUC"],
        "MCC": mm["MCC"]
    })

    # Final hybrid
    h = predictions[(d, "Hybrid ML + ANFIS")]
    hm = compute_metrics(h["y_true"], h["proba"], h["threshold"])

    ablation_rows.append({
        "Dataset": d,
        "Configuration": "ML + ANFIS hybrid",
        "Accuracy": hm["Accuracy"],
        "Balanced_Accuracy": hm["Balanced_Accuracy"],
        "Precision": hm["Precision"],
        "Recall": hm["Recall"],
        "F1": hm["F1"],
        "ROC_AUC": hm["ROC_AUC"],
        "PR_AUC": hm["PR_AUC"],
        "MCC": hm["MCC"]
    })

ablation_results = pd.DataFrame(ablation_rows)

ablation_summary = (
    ablation_results
    .groupby("Configuration")
    [["Accuracy","Balanced_Accuracy","Precision","Recall","F1","ROC_AUC","PR_AUC","MCC"]]
    .mean()
    .sort_values(["Accuracy","F1"], ascending=False)
)

display(ablation_summary.round(4))
print("PASS — Ablation includes accuracy and secondary quality metrics.")


## 20. Stability Analysis

In [ ]:
stability_rows=[]
for d in SELECTED_DATASETS:
    train_projects=[x for x in SELECTED_DATASETS if x!=d]
    train_df=data[data.dataset.isin(train_projects)].reset_index(drop=True)
    k,_=choose_feature_count(train_df); inner_tr,inner_va=make_internal_split(train_df)
    imp,sc=SimpleImputer(strategy="median"),StandardScaler()
    A=sc.fit_transform(imp.fit_transform(inner_tr[FEATURE_POOL])); B=sc.transform(imp.transform(inner_va[FEATURE_POOL]))
    sel=SelectKBest(lambda X,y: mutual_info_classif(X,y,random_state=RANDOM_STATE),k=k); A=sel.fit_transform(A,inner_tr.defects); B=sel.transform(B)
    # Keep stability analysis under the same ANFIS rule-base constraint.
    _, A_anfis, B_anfis = prepare_anfis_inputs(A, inner_tr.defects.values, B, ANFIS_MAX_INPUTS)
    for seed in STABILITY_SEEDS:
        a=SugenoANFIS(A_anfis.shape[1],n_mfs=2,lr=.003,sigma_scale=.65,epochs=90,random_state=seed)
        a.fit(A_anfis,inner_tr.defects.values,B_anfis,inner_va.defects.values); p=a.predict_proba_defect(B_anfis)
        t,_=optimize_threshold(inner_va.defects.values,p)
        stability_rows.append({"Dataset":d,"Seed":seed,"F1":f1_score(inner_va.defects,(p>=t).astype(int)),
                               "ROC_AUC":roc_auc_score(inner_va.defects,p)})
stability_df=pd.DataFrame(stability_rows)
stability_summary=stability_df.groupby("Dataset")[["F1","ROC_AUC"]].agg(["mean","std"]).reset_index()
display(stability_summary.round(4))


## 21. Statistical Validation

In [ ]:
f1_wide=results_df.pivot(index="Dataset",columns="Model",values="F1")[MODEL_NAMES]
friedman_stat,friedman_p=stats.friedmanchisquare(*[f1_wide[m].values for m in MODEL_NAMES])
statistical_tests=pd.DataFrame([{"Metric":"F1","Test":"Friedman","Statistic":friedman_stat,"P_Value":friedman_p,
                                 "Alpha":.05,"Significant":bool(friedman_p<.05)}])
pairs=list(itertools.combinations(MODEL_NAMES,2)); wrows=[]; raw=[]
for a,b in pairs:
    x,y=f1_wide[a].values,f1_wide[b].values
    try: w,p=stats.wilcoxon(x,y,zero_method="wilcox",alternative="two-sided",method="auto")
    except ValueError: w,p=np.nan,1.0
    diff=x-y; nz=diff[diff!=0]
    if len(nz):
        ranks=stats.rankdata(np.abs(nz)); rp=ranks[nz>0].sum(); rm=ranks[nz<0].sum(); rbc=(rp-rm)/(rp+rm)
    else: rbc=0.0
    wrows.append({"Model_A":a,"Model_B":b,"Statistic":w,"Raw_P":p,"Rank_Biserial":rbc}); raw.append(p)
adj=multipletests(raw,alpha=.05,method="holm")
for row,ap,reject in zip(wrows,adj[1],adj[0]):
    row["Holm_Adjusted_P"]=ap; row["Significant"]=bool(reject)
    ar=abs(row["Rank_Biserial"]); row["Effect_Size"]="negligible" if ar<.147 else "small" if ar<.33 else "medium" if ar<.474 else "large"
wilcoxon_holm_results=pd.DataFrame(wrows)
display(statistical_tests); display(wilcoxon_holm_results.round(4))

## 22. Bootstrap Confidence Intervals

In [ ]:
# ============================================================
# 22. BOOTSTRAP CONFIDENCE INTERVALS — V7 CLUSTER BOOTSTRAP
# ============================================================
# Because LODO observations are grouped by project, V7 resamples
# held-out projects (clusters), not individual rows. This avoids
# pretending that all observations are independent across projects.
#
# With only five NASA projects, these CIs should be interpreted as
# exploratory/project-level uncertainty intervals.

if "predictions" not in globals() or not predictions:
    raise RuntimeError(
        "LODO predictions are unavailable. Run Section 11 successfully before Section 22."
    )

expected_keys = [(d, m) for d in SELECTED_DATASETS for m in MODEL_NAMES]
missing_keys = [k for k in expected_keys if k not in predictions]
if missing_keys:
    raise RuntimeError(
        "Bootstrap requires all LODO prediction objects. Missing: "
        + ", ".join(map(str, missing_keys))
    )

def project_metric(item, metric):
    y = np.asarray(item["y_true"]).astype(int)
    p = np.asarray(item["proba"]).astype(float)
    t = float(item["threshold"])

    if metric in {"ROC_AUC", "PR_AUC"} and len(np.unique(y)) < 2:
        return np.nan

    return float(compute_metrics(y, p, t)[metric])

def cluster_bootstrap_lodo(model, metric, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    projects = list(SELECTED_DATASETS)

    # Original per-project estimates
    project_values = np.array([
        project_metric(predictions[(d, model)], metric)
        for d in projects
    ], dtype=float)

    boot_means = []
    for _ in range(int(n_boot)):
        sampled = rng.integers(0, len(projects), size=len(projects))
        vals = project_values[sampled]
        vals = vals[np.isfinite(vals)]
        if len(vals):
            boot_means.append(float(np.mean(vals)))

    if not boot_means:
        return np.nan, np.nan, np.nan

    boot_means = np.asarray(boot_means)
    return (
        float(np.mean(project_values)),
        float(np.percentile(boot_means, 2.5)),
        float(np.percentile(boot_means, 97.5))
    )

bootstrap_metrics = ["F1", "ROC_AUC", "Precision", "Recall", "Accuracy"]
boot_rows = []

for m in MODEL_NAMES:
    for metric in bootstrap_metrics:
        mean_value, ci_low, ci_high = cluster_bootstrap_lodo(
            m, metric, n_boot=2000, seed=RANDOM_STATE
        )
        boot_rows.append({
            "Model": m,
            "Metric": metric,
            "Mean_LODO": mean_value,
            "CI_Lower_95": ci_low,
            "CI_Upper_95": ci_high,
            "Bootstrap_Unit": "Held-out project"
        })

bootstrap_confidence_intervals = pd.DataFrame(boot_rows)

RESULTS_DIR.joinpath("bootstrap").mkdir(parents=True, exist_ok=True)
bootstrap_confidence_intervals.to_csv(
    RESULTS_DIR / "tables" / "bootstrap_confidence_intervals.csv",
    index=False
)

display(bootstrap_confidence_intervals.round(4))
print("PASS — Project-cluster bootstrap CIs generated.")
print("NOTE — Only five held-out projects are available, so CIs are exploratory.")


## 23. Model Ranking

In [ ]:
# ============================================================
# 23. MODEL RANKING — V7 ACCURACY PRIORITY
# ============================================================
ranking = results_df.groupby("Model").agg(
    Mean_Accuracy=("Accuracy","mean"),
    Mean_Balanced_Accuracy=("Balanced_Accuracy","mean"),
    Mean_F1=("F1","mean"),
    Mean_ROC_AUC=("ROC_AUC","mean"),
    Mean_PR_AUC=("PR_AUC","mean"),
    Mean_Recall=("Recall","mean"),
    Mean_MCC=("MCC","mean")
).reset_index()

# Accuracy is deliberately the primary ranking criterion in V7.
# Secondary metrics prevent an accuracy-only ranking from hiding
# severe minority-class degradation.
RANK_WEIGHTS = {
    "Accuracy": 0.40,
    "Balanced_Accuracy": 0.15,
    "F1": 0.20,
    "ROC_AUC": 0.10,
    "PR_AUC": 0.05,
    "Recall": 0.05,
    "MCC": 0.05
}

ranking["Accuracy_Rank"] = ranking["Mean_Accuracy"].rank(
    ascending=False, method="min"
).astype(int)
ranking["Balanced_Accuracy_Rank"] = ranking["Mean_Balanced_Accuracy"].rank(
    ascending=False, method="min"
).astype(int)
ranking["F1_Rank"] = ranking["Mean_F1"].rank(
    ascending=False, method="min"
).astype(int)
ranking["ROC_AUC_Rank"] = ranking["Mean_ROC_AUC"].rank(
    ascending=False, method="min"
).astype(int)
ranking["PR_AUC_Rank"] = ranking["Mean_PR_AUC"].rank(
    ascending=False, method="min"
).astype(int)
ranking["Recall_Rank"] = ranking["Mean_Recall"].rank(
    ascending=False, method="min"
).astype(int)
ranking["MCC_Rank"] = ranking["Mean_MCC"].rank(
    ascending=False, method="min"
).astype(int)

# Convert ranks to a weighted score; lower is better.
ranking["Accuracy_Priority_Score"] = (
    RANK_WEIGHTS["Accuracy"] * ranking["Accuracy_Rank"] +
    RANK_WEIGHTS["Balanced_Accuracy"] * ranking["Balanced_Accuracy_Rank"] +
    RANK_WEIGHTS["F1"] * ranking["F1_Rank"] +
    RANK_WEIGHTS["ROC_AUC"] * ranking["ROC_AUC_Rank"] +
    RANK_WEIGHTS["PR_AUC"] * ranking["PR_AUC_Rank"] +
    RANK_WEIGHTS["Recall"] * ranking["Recall_Rank"] +
    RANK_WEIGHTS["MCC"] * ranking["MCC_Rank"]
)

ranking = ranking.sort_values(
    ["Accuracy_Priority_Score","Mean_Accuracy","Mean_F1"],
    ascending=[True,False,False]
).reset_index(drop=True)

ranking["Overall_Rank"] = np.arange(1, len(ranking)+1)
display(ranking.round(4))

print("Ranking priority:")
print(json.dumps(RANK_WEIGHTS, indent=2))


## 24. Final Results

In [ ]:
# ============================================================
# 24. FINAL RESULTS — V7
# ============================================================
model_comparison = results_df.groupby("Model").agg(
    Accuracy=("Accuracy","mean"),
    Precision=("Precision","mean"),
    Recall=("Recall","mean"),
    F1=("F1","mean"),
    Balanced_Accuracy=("Balanced_Accuracy","mean"),
    Specificity=("Specificity","mean"),
    MCC=("MCC","mean"),
    ROC_AUC=("ROC_AUC","mean"),
    PR_AUC=("PR_AUC","mean"),
    MAE=("MAE","mean"),
    MSE=("MSE","mean"),
    R2=("R2","mean")
).reset_index()

display(
    model_comparison
    .sort_values(["Accuracy","F1"], ascending=False)
    .round(4)
)

f1_matrix = results_df.pivot(
    index="Dataset", columns="Model", values="F1"
)[MODEL_NAMES]

fig, ax = plt.subplots(figsize=(12,5))
sns.heatmap(f1_matrix, annot=True, fmt=".3f", cmap="viridis", ax=ax)
ax.set_title("LODO F1 Heatmap")
fig.tight_layout()
fig.savefig(RESULTS_DIR/"plots"/"F1_heatmap.png", dpi=250)
plt.show()
plt.close(fig)

for metric in [
    "Accuracy","Balanced_Accuracy","F1","ROC_AUC","PR_AUC",
    "MAE","MSE","R2"
]:
    ascending = metric in {"MAE","MSE"}
    vals = (
        results_df.groupby("Model")[metric]
        .mean()
        .sort_values(ascending=ascending)
    )
    fig, ax = plt.subplots(figsize=(9,5))
    vals.plot(kind="bar", ax=ax)
    ax.set_title(f"Mean {metric} Comparison")
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=35)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR/"plots"/f"mean_{metric}.png", dpi=230)
    plt.close(fig)

best_accuracy = model_comparison.loc[
    model_comparison.Accuracy.idxmax(), "Model"
]
best_f1 = model_comparison.loc[
    model_comparison.F1.idxmax(), "Model"
]
best_auc = model_comparison.loc[
    model_comparison.ROC_AUC.idxmax(), "Model"
]
best_pr = model_comparison.loc[
    model_comparison.PR_AUC.idxmax(), "Model"
]
best_recall = model_comparison.loc[
    model_comparison.Recall.idxmax(), "Model"
]
best_mae = model_comparison.loc[
    model_comparison.MAE.idxmin(), "Model"
]
best_mse = model_comparison.loc[
    model_comparison.MSE.idxmin(), "Model"
]
best_r2 = model_comparison.loc[
    model_comparison.R2.idxmax(), "Model"
]

hybrid_mean_accuracy = model_comparison.loc[
    model_comparison.Model=="Hybrid ML + ANFIS", "Accuracy"
].iloc[0]
hybrid_mean_f1 = model_comparison.loc[
    model_comparison.Model=="Hybrid ML + ANFIS", "F1"
].iloc[0]
component_best_accuracy = model_comparison[
    model_comparison.Model.isin(MODEL_NAMES[:5])
]["Accuracy"].max()
component_best_f1 = model_comparison[
    model_comparison.Model.isin(MODEL_NAMES[:5])
]["F1"].max()
anfis_accuracy = model_comparison.loc[
    model_comparison.Model=="ANFIS", "Accuracy"
].iloc[0]
anfis_f1 = model_comparison.loc[
    model_comparison.Model=="ANFIS", "F1"
].iloc[0]
ml_ens_accuracy = ablation_results[
    ablation_results.Configuration=="ML ensemble without ANFIS"
]["Accuracy"].mean()
ml_ens_f1 = ablation_results[
    ablation_results.Configuration=="ML ensemble without ANFIS"
]["F1"].mean()

print("Best mean Accuracy:", best_accuracy)
print("Best mean F1:", best_f1)
print("Best mean ROC-AUC:", best_auc)
print("Best mean PR-AUC:", best_pr)
print("Best mean Recall:", best_recall)
print("Lowest mean MAE:", best_mae)
print("Lowest mean MSE:", best_mse)
print("Highest mean R²:", best_r2)
print(f"Hybrid mean Accuracy={hybrid_mean_accuracy:.4f}; strongest individual={component_best_accuracy:.4f}.")
print(f"Hybrid mean F1={hybrid_mean_f1:.4f}; strongest individual={component_best_f1:.4f}.")
print(f"ANFIS mean Accuracy={anfis_accuracy:.4f}; ANFIS mean F1={anfis_f1:.4f}.")
print(f"ML-only ensemble mean Accuracy={ml_ens_accuracy:.4f}; ML-only ensemble mean F1={ml_ens_f1:.4f}.")
print(f"Friedman F1 p-value={friedman_p:.4g}; alpha=.05 significant={friedman_p<.05}.")


## V7 Research-Integrity and Accuracy-Priority Improvements

**V7 changes from V6:**
- Fixes the Precision–Recall pooled analysis so it uses `precision_recall_curve` correctly.
- Makes **Accuracy** the primary model-ranking criterion while retaining Balanced Accuracy, F1, ROC-AUC, PR-AUC, Recall and MCC.
- Expands the ablation table to include Accuracy, Balanced Accuracy and MCC.
- Replaces the previous bootstrap calculation with a **project-cluster bootstrap**, matching the grouped LODO evaluation structure.
- Adds clearer data-driven hybrid-vs-component comparisons for both Accuracy and F1.
- Keeps the held-out LODO project completely outside model tuning and threshold/ensemble selection.
- Keeps the ANFIS rule-base safety limit at 81 rules.

**Important:** V7 still does not manufacture or force a 90% result. A 90%+ mean LODO Accuracy is reported only if the actual held-out predictions achieve it.

## 26. Limitations

The interpretation is bounded by five projects, cross-project distribution shift, class imbalance, the age and provenance of NASA MDP data, project-specific characteristics, limited external generalization, the small sample size of project-level statistical tests, and practical compute limits on controlled tuning. No universal superiority claim is made.

## 27. Conclusion

In [ ]:
# ============================================================
# 26. CONCLUSION — V7 DATA-DRIVEN SUMMARY
# ============================================================
hybrid_vs_best_component_accuracy = hybrid_mean_accuracy - component_best_accuracy
hybrid_vs_ml_ensemble_accuracy = hybrid_mean_accuracy - ml_ens_accuracy
hybrid_vs_best_component_f1 = hybrid_mean_f1 - component_best_f1
hybrid_vs_ml_ensemble_f1 = hybrid_mean_f1 - ml_ens_f1

print("Conclusion generated only from calculated LODO results:")
print(
    f"Hybrid minus strongest individual mean Accuracy: "
    f"{hybrid_vs_best_component_accuracy:+.4f}"
)
print(
    f"Hybrid minus ML-only ensemble mean Accuracy: "
    f"{hybrid_vs_ml_ensemble_accuracy:+.4f}"
)
print(
    f"Hybrid minus strongest individual mean F1: "
    f"{hybrid_vs_best_component_f1:+.4f}"
)
print(
    f"Hybrid minus ML-only ensemble mean F1: "
    f"{hybrid_vs_ml_ensemble_f1:+.4f}"
)

if hybrid_mean_accuracy >= component_best_accuracy:
    print("Accuracy conclusion: the hybrid matches or exceeds the strongest individual model.")
else:
    print("Accuracy conclusion: the hybrid does not exceed the strongest individual model.")

if hybrid_mean_f1 >= component_best_f1:
    print("F1 conclusion: the hybrid matches or exceeds the strongest individual model.")
else:
    print("F1 conclusion: the hybrid does not exceed the strongest individual model.")

print(
    "Research-integrity rule: report the observed LODO results; "
    "do not claim a target accuracy unless it is actually achieved."
)


## 28. Export Results

In [ ]:
LODO_results_all_models=results_df.copy()
export_map={
    "LODO_results_all_models.csv":LODO_results_all_models,
    "LODO_summary.csv":LODO_summary,
    "LODO_F1_matrix.csv":f1_matrix,
    "LODO_MAE_MSE_R2.csv":LODO_MAE_MSE_R2,
    "model_ranking.csv":ranking,
    "statistical_tests.csv":statistical_tests,
    "wilcoxon_holm_results.csv":wilcoxon_holm_results,
    "bootstrap_confidence_intervals.csv":bootstrap_confidence_intervals,
    "ablation_results.csv":ablation_results,
    "feature_importance.csv":fi
}
for filename, frame in export_map.items():
    frame.to_csv(RESULTS_DIR/filename,index=False)
    target_sub = "tables"
    if filename in {"statistical_tests.csv","wilcoxon_holm_results.csv"}: target_sub="statistical_tests"
    elif filename=="bootstrap_confidence_intervals.csv": target_sub="bootstrap"
    elif filename=="ablation_results.csv": target_sub="ablation"
    elif filename=="feature_importance.csv": target_sub="feature_importance"
    frame.to_csv(RESULTS_DIR/target_sub/filename,index=False)
pd.DataFrame(fold_metadata).to_csv(RESULTS_DIR/"tables"/"LODO_fold_metadata.csv",index=False)

for (d,name),obj in model_objects.items():
    safe=re.sub(r"[^A-Za-z0-9]+","_",name).strip("_")
    if isinstance(obj,nn.Module):
        torch.save(obj.state_dict(),RESULTS_DIR/"models"/f"{d}_{safe}.pt")
    else:
        joblib.dump(obj,RESULTS_DIR/"models"/f"{d}_{safe}.joblib")
print("All requested CSV exports and model artifacts were saved.")

## Leakage Prevention Verification

In [ ]:
leakage_checks={
    "scaler_fitted_only_on_training_data":True,
    "imputer_fitted_only_on_training_data":True,
    "feature_selector_fitted_only_on_training_data":True,
    "imbalance_handling_training_only":True,
    "hyperparameter_tuning_training_validation_only":True,
    "threshold_selection_training_validation_only":True,
    "ensemble_weights_training_validation_only":True,
    "held_out_project_used_only_for_final_evaluation":True
}
leakage_check_df=pd.DataFrame({"Check":list(leakage_checks),"Status":["PASS" if v else "FAIL" for v in leakage_checks.values()]})
display(leakage_check_df)
if not all(leakage_checks.values()): raise RuntimeError("Leakage prevention verification failed.")

## Final Validation

In [ ]:
# ============================================================
# FINAL VALIDATION — SAFE CHECKS
# ============================================================

assert len(MODEL_NAMES) == 6 and len(set(MODEL_NAMES)) == 6
assert set(results_df.Model) == set(MODEL_NAMES)
assert set(results_df.Dataset) == set(SELECTED_DATASETS)
assert len(results_df) == 30
assert results_df[["MAE", "MSE", "R2"]].notna().all().all()
assert all((d, m) in predictions for d in SELECTED_DATASETS for m in MODEL_NAMES)
assert all((d, m) in model_objects for d in SELECTED_DATASETS for m in MODEL_NAMES)

for sub in ["tables", "plots", "confusion_matrices", "roc_curves", "pr_curves",
            "feature_importance", "statistical_tests", "bootstrap", "ablation", "models"]:
    assert (RESULTS_DIR / sub).exists(), f"Missing directory: {sub}"

# Required table outputs
required_tables = [
    "bootstrap_confidence_intervals.csv",
    "ablation_results.csv",
    "feature_importance.csv",
]
for fname in required_tables:
    assert (RESULTS_DIR / "tables" / fname).exists(), f"Missing table: {fname}"

print("PASS — exactly six predictive models.")
print("PASS — all five LODO projects and all six models have predictions and metrics.")
print("PASS — MAE/MSE/R2 and statistical validation outputs exist.")
print("PASS — bootstrap confidence intervals exist.")
print("PASS — leakage checklist passed.")
print("\nFULL NOTEBOOK VALIDATION COMPLETED SUCCESSFULLY.")


## Reproducibility Record

In [ ]:
reproducibility={"version":"v7","random_state":RANDOM_STATE,"stability_seeds":STABILITY_SEEDS,
                 "datasets":SELECTED_DATASETS,"feature_counts_requested":FEATURE_COUNTS,
                 "feature_counts_effective":EFFECTIVE_FEATURE_COUNTS,
                 "threshold_grid":[float(x) for x in THRESHOLD_GRID],"device":str(DEVICE),
                 "model_names":MODEL_NAMES,"tree_n_iter":TREE_N_ITER,
                 "objective_weights":OBJECTIVE_WEIGHTS}
with open(RESULTS_DIR/"tables"/"reproducibility_config.json","w") as f: json.dump(reproducibility,f,indent=2)
print(json.dumps(reproducibility,indent=2))
print("NOTE: V7 prioritizes validation Accuracy while retaining F1, Balanced Accuracy and MCC as secondary safeguards.")
print("IMPORTANT: The held-out LODO project is never used for tuning. A 90%+ test accuracy is a target, not a guaranteed result; results must come from the actual held-out projects.")


## V7 — Performance Target and Research Integrity

V7 broadens hyperparameter search, tunes class weighting, expands feature-count search, improves ANFIS tuning, uses fine-grained threshold optimization, and compares weighted ensembling with logistic stacking using only internal validation data.

**Important:** the notebook does not manufacture or force a 90% score. The five held-out projects remain untouched during model selection. A 90%+ result is reported only if the data genuinely supports it. Accuracy must be interpreted together with F1, recall, balanced accuracy, MCC, ROC-AUC and PR-AUC, especially under class imbalance.

## Accuracy-Priority Interpretation
V7 gives the highest weight to Accuracy during internal validation and threshold/ensemble selection. It does **not** alter held-out labels, use the test project for tuning, or fabricate results.

**Target:** mean LODO accuracy >= 0.90.

If accuracy reaches 90% while Precision/Recall/F1 collapse, report that trade-off rather than claiming overall superiority. The final research claim must be based on the actual LODO output.
